# Game Intelligence AI — Stage 5: Source Registry, License Audit & Code Inventory

**Purpose:** inspect selected public GitHub repositories, pin exact commit SHAs, discover license/notice files, inventory source-code candidates, exclude assets and unsuitable files, and create a provenance audit.

## Critical legal and technical boundary
- This notebook is an **audit and inventory tool**, not a legal opinion.
- It does not assume that every file inherits a repository-level license.
- It excludes art, audio, fonts, models, videos, and other binary assets.
- It does not automatically turn raw source code into instruction/answer training examples.
- Actual code extraction is disabled until a human explicitly approves a repository at an exact commit SHA after reviewing its license evidence.
- Never approve a source merely because its GitHub page is public.

## 1. Runtime setup

Uses Python standard library plus `requests`. No Google Drive, no GPU, and no model downloads are required. Colab `/content` is temporary; download the audit outputs before ending the session.

In [ ]:
from pathlib import Path
import json, csv, re, hashlib, datetime, time, urllib.parse
from collections import Counter
import requests

WORK = Path("/content/game_intelligence_stage5")
OUT = WORK / "output"
RAW = WORK / "approved_raw_code"
OUT.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)

GITHUB_API = "https://api.github.com"
SESSION = requests.Session()
SESSION.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "GameIntelligenceAI-Dataset-Audit/1.0"
})

print("Workspace:", WORK)
print("No GPU or Google Drive is needed for this stage.")

## 2. Source registry

The first two candidates are:
- `godotengine/godot-demo-projects`
- `phaserjs/examples`

The registry stores why each source is interesting, what needs special care, and its current audit state. The Phaser examples README says the source code is MIT but assets are not; the repository also has package metadata and many asset-specific license files, so human review remains mandatory. The Godot demos have a root MIT license and additional nested license/notice files, so file-level exclusions are still required.

In [ ]:
SOURCE_REGISTRY = [
    {
        "repo": "godotengine/godot-demo-projects",
        "purpose": "Godot 2D/3D demonstrations and project patterns",
        "code_extensions": [".gd", ".cs", ".gdshader"],
        "special_care": "Exclude assets and inspect nested license/notice files; do not assume root license overrides file-specific notices.",
        "review_status": "pending_human_review"
    },
    {
        "repo": "phaserjs/examples",
        "purpose": "Phaser JavaScript/TypeScript game examples",
        "code_extensions": [".js", ".mjs", ".cjs", ".ts"],
        "special_care": "README states source code MIT and assets are not; package.json declares ISC. Human must resolve metadata/licensing scope before approval. Exclude public/assets and all media.",
        "review_status": "pending_human_review"
    }
]

registry_path = OUT / "source_registry.json"
registry_path.write_text(json.dumps(SOURCE_REGISTRY, indent=2), encoding="utf-8")
print(json.dumps(SOURCE_REGISTRY, indent=2))

## 3. Human approval gate — leave empty for the first run

Do not change this cell until you have reviewed the source's license text, repository notices, and any nested file-level licenses. Approval must be tied to the exact commit SHA reported by the audit. This prevents silently ingesting a later changed revision.

In [ ]:
# Intentionally empty on first run.
# After reviewing the generated audit, an expert/human may add an entry like:
# APPROVED_EXACT_COMMITS = {"owner/repo": "full_40_character_commit_sha"}
#
# Do not approve Phaser until the README MIT statement and package metadata
# difference have been reviewed for the specific code files you intend to use.
APPROVED_EXACT_COMMITS = {}
MAX_FILES_TO_EXTRACT_PER_REPO = 100
MAX_FILE_BYTES = 200_000

## 4. GitHub API helpers

In [ ]:
def github_get(url, timeout=30):
    response = SESSION.get(url, timeout=timeout)
    if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
        raise RuntimeError("GitHub API rate limit reached. Wait for reset or use a GitHub token stored privately in Colab Secrets.")
    response.raise_for_status()
    return response.json()

def raw_url(repo, commit_sha, path):
    encoded_path = urllib.parse.quote(path, safe="/")
    return f"https://raw.githubusercontent.com/{repo}/{commit_sha}/{encoded_path}"

def is_probably_generated_or_vendor(path):
    low = path.lower()
    blocked_parts = [
        "node_modules/", "vendor/", "third_party/", "third-party/",
        "generated/", "dist/", "build/", ".git/", "examples/assets/",
        "public/assets/", "assets/", "fonts/", "audio/", "sounds/",
        "textures/", "models/", "sprites/", "images/", "videos/"
    ]
    if any(part in low for part in blocked_parts):
        return True
    name = path.rsplit("/", 1)[-1].lower()
    return name.endswith((".min.js", ".bundle.js", ".generated.cs"))

def is_text_code_candidate(path, extensions):
    suffix = Path(path).suffix.lower()
    return suffix in set(extensions) and not is_probably_generated_or_vendor(path)

def license_ancestors(path, license_paths):
    parent_parts = path.split("/")[:-1]
    ancestors = [""]
    for i in range(1, len(parent_parts) + 1):
        ancestors.append("/".join(parent_parts[:i]))
    matches = []
    for lic in license_paths:
        lic_parent = lic.rsplit("/", 1)[0] if "/" in lic else ""
        if lic_parent in ancestors:
            matches.append(lic)
    return sorted(matches, key=lambda x: (x.count("/"), len(x)))

## 5. Inspect repositories and pin exact commits

This reads repository metadata, resolves the default branch to a commit SHA, and downloads the recursive Git tree metadata. It does not download source-code contents yet.

In [ ]:
repo_audits = []
for source in SOURCE_REGISTRY:
    repo = source["repo"]
    print("\\nInspecting:", repo)
    try:
        meta = github_get(f"{GITHUB_API}/repos/{repo}")
        branch = meta["default_branch"]
        commit_data = github_get(f"{GITHUB_API}/repos/{repo}/commits/{branch}")
        commit_sha = commit_data["sha"]
        tree_data = github_get(f"{GITHUB_API}/repos/{repo}/git/trees/{commit_sha}?recursive=1")
        tree = tree_data.get("tree", [])
        audit = {
            **source,
            "default_branch": branch,
            "commit_sha": commit_sha,
            "commit_url": commit_data.get("html_url"),
            "repo_url": meta.get("html_url"),
            "repo_license_spdx": (meta.get("license") or {}).get("spdx_id"),
            "repo_pushed_at": meta.get("pushed_at"),
            "tree_truncated": tree_data.get("truncated", False),
            "tree_entries": tree,
            "api_error": None
        }
        print("Pinned:", commit_sha, "| entries:", len(tree), "| truncated:", audit["tree_truncated"])
    except Exception as exc:
        audit = {**source, "api_error": str(exc), "tree_entries": []}
        print("ERROR:", exc)
    repo_audits.append(audit)

## 6. License and notice discovery

License-like files are inventoried at every folder level. Their presence is a signal for manual inspection, not an automatic conclusion. A nested license may apply to only one sample or asset.

In [ ]:
for audit in repo_audits:
    tree = audit.get("tree_entries", [])
    license_paths = [
        item["path"] for item in tree
        if item.get("type") == "blob"
        and re.search(r"(^|/)(license|licence|copying|notice)(\\.|$)|license|licence|copying|notice", item["path"], re.I)
    ]
    readme_paths = [
        item["path"] for item in tree
        if item.get("type") == "blob"
        and item["path"].lower().split("/")[-1].startswith("readme")
    ]
    audit["license_paths"] = license_paths
    audit["readme_paths"] = readme_paths
    print("\\n", audit["repo"])
    print("License/notice files:", len(license_paths))
    for p in license_paths[:35]:
        print(" -", p)

## 7. Inspect key root license evidence

Fetches only small root-level legal/documentation files (LICENSE variants and README). These are stored as evidence snippets in the audit report. They are not treated as proof that every asset is licensed for reuse.

In [ ]:
evidence_records = []
for audit in repo_audits:
    if audit.get("api_error") or not audit.get("commit_sha"):
        continue
    repo = audit["repo"]
    paths = set(audit.get("license_paths", []))
    paths.update(p for p in audit.get("readme_paths", []) if p.count("/") == 0)
    # Root legal evidence only in this cell; nested notices are listed for later review.
    selected = [p for p in paths if p.count("/") == 0][:8]
    for path in selected:
        try:
            url = raw_url(repo, audit["commit_sha"], path)
            r = SESSION.get(url, timeout=25)
            r.raise_for_status()
            body = r.text[:30000]
            evidence_records.append({
                "repo": repo, "commit_sha": audit["commit_sha"],
                "path": path, "url": url,
                "sha256": hashlib.sha256(r.content).hexdigest(),
                "text_excerpt": body[:6000]
            })
            print("\\n---", repo, path, "---\\n", body[:1200])
        except Exception as exc:
            evidence_records.append({"repo": repo, "path": path, "error": str(exc)})

## 8. Build source-code candidate inventory

Only source-code extensions declared in the registry are considered. Media, art, fonts, 3D models, audio, video, minified bundles, vendor folders, and common generated folders are excluded. A candidate is **not approved for training** just because it appears in this inventory.

In [ ]:
candidate_rows = []
for audit in repo_audits:
    if audit.get("api_error"):
        continue
    license_paths = audit.get("license_paths", [])
    for item in audit.get("tree_entries", []):
        if item.get("type") != "blob":
            continue
        path = item["path"]
        if not is_text_code_candidate(path, audit["code_extensions"]):
            continue
        ancestors = license_ancestors(path, license_paths)
        candidate_rows.append({
            "repo": audit["repo"],
            "commit_sha": audit["commit_sha"],
            "path": path,
            "size_bytes": item.get("size"),
            "blob_sha": item.get("sha"),
            "nearest_license_or_notice_files": ancestors,
            "candidate_status": "needs_human_file_level_license_review",
            "approved_for_extraction": False
        })

print("Code candidates:", len(candidate_rows))
print("By repo:", dict(Counter(x["repo"] for x in candidate_rows)))
for row in candidate_rows[:25]:
    print(row["repo"], row["path"], "license evidence:", row["nearest_license_or_notice_files"][-3:])

## 9. Optional extraction — only exact human-approved commits

Default behavior is **no extraction**. To enable it:
1. Read the audit report and inspect the linked license/notice files.
2. Confirm the exact source files are covered by the license.
3. Add the exact 40-character commit SHA to `APPROVED_EXACT_COMMITS` in Section 3.
4. Re-run the notebook.

Extracted files are stored as a raw code corpus with provenance. They are not instruction/answer examples and must not be passed directly to the QLoRA SFT trainer.

In [ ]:
extraction_log = []
for audit in repo_audits:
    repo = audit.get("repo")
    sha = audit.get("commit_sha")
    if not repo or not sha or APPROVED_EXACT_COMMITS.get(repo) != sha:
        continue

    repo_rows = [r for r in candidate_rows if r["repo"] == repo and r["commit_sha"] == sha]
    repo_rows = repo_rows[:MAX_FILES_TO_EXTRACT_PER_REPO]
    extracted = 0

    for row in repo_rows:
        size = row.get("size_bytes")
        if size is not None and size > MAX_FILE_BYTES:
            extraction_log.append({**row, "extraction_status": "skipped_too_large"})
            continue
        try:
            url = raw_url(repo, sha, row["path"])
            response = SESSION.get(url, timeout=30)
            response.raise_for_status()
            content = response.content
            if len(content) > MAX_FILE_BYTES or b"\\x00" in content:
                extraction_log.append({**row, "extraction_status": "skipped_binary_or_too_large"})
                continue
            text_content = content.decode("utf-8")
            digest = hashlib.sha256(content).hexdigest()
            safe_repo = repo.replace("/", "__")
            dest = RAW / safe_repo / sha / row["path"]
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(text_content, encoding="utf-8")
            extraction_log.append({
                **row, "extraction_status": "extracted_after_human_repo_approval",
                "raw_url": url, "content_sha256": digest, "local_path": str(dest)
            })
            extracted += 1
        except Exception as exc:
            extraction_log.append({**row, "extraction_status": "error", "error": str(exc)})
    print(repo, "extracted files:", extracted)

if not extraction_log:
    print("No files extracted. This is expected until exact commits are explicitly approved.")

## 10. Export audit files

Outputs:
- `stage5_repository_audit.json`: repository metadata, pinned commits, license paths, README/license evidence, candidate inventory, and extraction log.
- `stage5_code_candidates.csv`: reviewable list of source-code candidates.
- `stage5_extraction_log.json`: extraction decisions (normally empty on first run).

The outputs are small. Raw extracted code, if explicitly enabled, stays in the Colab workspace and should be handled according to each source license.

In [ ]:
report = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "stage": "5_source_registry_license_audit_code_inventory",
    "repositories": [
        {k: v for k, v in audit.items() if k != "tree_entries"}
        for audit in repo_audits
    ],
    "license_evidence": evidence_records,
    "candidate_count": len(candidate_rows),
    "candidate_counts_by_repo": dict(Counter(x["repo"] for x in candidate_rows)),
    "candidate_inventory": candidate_rows,
    "approved_exact_commits": APPROVED_EXACT_COMMITS,
    "extraction_log": extraction_log,
    "training_dataset_created": False,
    "training_authorized": False,
    "warnings": [
        "Public availability is not permission to train or commercially reuse content.",
        "Repository-level licenses may not cover every file or third-party asset.",
        "Every selected file requires human review of applicable license and notices.",
        "Raw source code is not automatically a supervised instruction/answer dataset.",
        "This audit does not constitute legal advice."
    ]
}

audit_path = OUT / "stage5_repository_audit.json"
audit_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

csv_path = OUT / "stage5_code_candidates.csv"
if candidate_rows:
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(candidate_rows[0].keys()))
        writer.writeheader()
        for row in candidate_rows:
            row_copy = dict(row)
            row_copy["nearest_license_or_notice_files"] = " | ".join(row_copy["nearest_license_or_notice_files"])
            writer.writerow(row_copy)
else:
    csv_path.write_text("repo,commit_sha,path,candidate_status\\n", encoding="utf-8")

extract_path = OUT / "stage5_extraction_log.json"
extract_path.write_text(json.dumps(extraction_log, ensure_ascii=False, indent=2), encoding="utf-8")

print("Audit:", audit_path)
print("Candidates CSV:", csv_path)
print("Extraction log:", extract_path)
print("Repositories inspected:", len(repo_audits))
print("Code candidates:", len(candidate_rows))
print("Files extracted:", sum(x.get("extraction_status") == "extracted_after_human_repo_approval" for x in extraction_log))

## 11. Download audit outputs

In [ ]:
try:
    from google.colab import files
    files.download(str(audit_path))
    files.download(str(csv_path))
    files.download(str(extract_path))
except Exception as exc:
    print("Colab download helper unavailable:", repr(exc))
    print("Files remain in:", OUT)

## Stage 5 acceptance criteria

Do not move to training until:
1. Repository commit SHAs are pinned.
2. Root and nested license/notice files are reviewed.
3. File-level license coverage is confirmed for every selected source file.
4. All assets and third-party materials without clear permission are excluded.
5. Extracted code has provenance and SHA-256 hashes.
6. A separate curation process transforms selected code into high-quality instruction/context/answer examples, with tests and human review.
7. Train/validation/test groups are assigned by project/source to prevent leakage.

**Expected first-run state:** repository inventory and audit report are created; extraction remains disabled; training remains unauthorized.